# SQL vs pandas : le guide pour comparer les deux

Notebook de l'article [SQL vs pandas : le guide pour comparer les deux](https://moncoachdata.com/blog/sql-vs-pandas/) (MonCoachData).

Chaque requête SQL (exécutée avec SQLite) est comparée à son équivalent pandas sur le jeu de données Titanic. Exécute les cellules **dans l'ordre**. Code testé en octobre 2026 avec Python 3.13 et pandas 3.

In [ ]:
import sys, sqlite3, pandas as pd

print("Python", sys.version.split()[0])
print("pandas", pd.__version__)
print("SQLite", sqlite3.sqlite_version)

## Charger le jeu de données Titanic dans pandas et dans SQL

In [ ]:
url = "https://raw.githubusercontent.com/moncoachdata/blog-data-science/main/sql-vs-pandas/titanic.csv"
titanic = pd.read_csv(url)

pd.set_option("display.max_rows", 8)
titanic.shape

In [ ]:
titanic.head()

In [ ]:
conn = sqlite3.connect(":memory:")
titanic.to_sql("titanic", conn, index=False)

def sql(requete):
    return pd.read_sql_query(requete, conn)

## Sélectionner des colonnes

In [ ]:
sql("""
SELECT pclass, fare, age, survived
FROM titanic
""")

In [ ]:
col_list = ['pclass', 'fare', 'age', 'survived']
titanic[col_list]

In [ ]:
titanic[['pclass', 'fare', 'age', 'survived']].equals(titanic[col_list])

In [ ]:
titanic.loc[:, ['pclass', 'fare', 'age', 'survived']].equals(titanic[col_list])

In [ ]:
titanic.loc[50:100, col_list]

## Filtrer des lignes (WHERE)

In [ ]:
sql("""
SELECT pclass, fare, age, survived
FROM titanic
WHERE sex = 'male' AND age >= 18
""")

In [ ]:
male_mask = titanic.sex == 'male'
age_mask = titanic.age >= 18
col_list = ['pclass', 'fare', 'age', 'survived']

titanic.loc[male_mask & age_mask, col_list]

In [ ]:
titanic.loc[~male_mask, 'sex'].value_counts()

In [ ]:
titanic.query("sex == 'male' and age >= 18")[col_list]

In [ ]:
print("Passagers dont l'âge est inconnu :", titanic.age.isna().sum())
print("Lignes avec age >= 18 :", (titanic.age >= 18).sum())
print("Lignes avec age < 18  :", (titanic.age < 18).sum())

## Trier (ORDER BY)

Avec pandas 3, modifier une tranche ne modifie jamais le DataFrame d'origine (copie sur écriture).

In [ ]:
sql("""
SELECT pclass, fare, age, survived
FROM titanic
WHERE sex = 'male' AND age >= 18
ORDER BY fare DESC, age
""")

In [ ]:
df_slice = titanic.loc[male_mask & age_mask, col_list]
df_slice.sort_values(['fare', 'age'], ascending=[False, True])

In [ ]:
df_slice = titanic.loc[male_mask & age_mask, col_list]
df_slice.loc[:, 'fare'] = 0

print("Tarif max dans la tranche :", df_slice.fare.max())
print("Tarif max dans titanic    :", titanic.fare.max())

## Limiter (LIMIT)

In [ ]:
sql("""
SELECT pclass, fare, age, survived
FROM titanic
WHERE sex = 'male' AND age >= 18
ORDER BY fare DESC, age
LIMIT 10
""")

In [ ]:
df_slice = titanic.loc[male_mask & age_mask, col_list]
df_slice.sort_values(['fare', 'age'], ascending=[False, True]).head(10)

In [ ]:
df_slice.nsmallest(10, ['fare', 'age'])

## Fonctions d'agrégation

In [ ]:
sql("""
SELECT MAX(fare), MIN(fare), AVG(age), SUM(survived)
FROM titanic
""")

In [ ]:
print(titanic.fare.agg(['max', 'min']))
print("Âge moyen :", titanic.age.mean())
print("Survivants :", titanic.survived.sum())

In [ ]:
print(titanic.age.mean().round(1))

In [ ]:
print("Médiane de l'âge :", titanic.age.median())
print("Écart type de l'âge :", round(titanic.age.std(), 2))
print("Variance de l'âge :", round(titanic.age.var(), 2))

In [ ]:
titanic.describe()

In [ ]:
titanic.describe(include="str")

## GROUP BY et groupby()

In [ ]:
sql("""
SELECT sex, pclass, COUNT(*), SUM(survived), AVG(age)
FROM titanic
GROUP BY sex, pclass
""")

In [ ]:
print(titanic.groupby(['sex', 'pclass']).survived.agg(['count', 'sum']))
print(titanic.groupby(['sex', 'pclass']).age.mean())

In [ ]:
new_df = titanic.groupby(['sex', 'pclass'])[['survived', 'age']]
new_df.agg(['count', 'sum', 'mean'])

In [ ]:
titanic.groupby(['sex', 'pclass']).agg(
    passagers=('survived', 'size'),
    survivants=('survived', 'sum'),
    age_moyen=('age', 'mean'),
)

## Valeurs manquantes dans un regroupement

SQL garde un groupe `NULL`, pandas l'écarte par défaut (`dropna=False` pour l'inclure).

In [ ]:
sql("""
SELECT embarked, COUNT(*)
FROM titanic
GROUP BY embarked
""")

In [ ]:
titanic.groupby('embarked').size()

In [ ]:
titanic.groupby('embarked', dropna=False).size()

## Vérifier que SQL et pandas donnent le même résultat

In [ ]:
import numpy as np

def identique(a, b):
    a = a.reset_index(drop=True).copy()
    b = b.reset_index(drop=True).copy()
    b.columns = a.columns
    return np.allclose(a.to_numpy(dtype=float), b.to_numpy(dtype=float), equal_nan=True)

req = """
SELECT pclass, fare, age, survived
FROM titanic
WHERE sex = 'male' AND age >= 18
ORDER BY fare DESC, age
LIMIT 10
"""
df_slice = titanic.loc[male_mask & age_mask, ['pclass', 'fare', 'age', 'survived']]
resultat_pandas = df_slice.sort_values(['fare', 'age'], ascending=[False, True]).head(10)

identique(sql(req), resultat_pandas)

## Exercices : corrections des 6 problèmes

Essaie d'abord de résoudre chaque problème toi-même, puis compare avec la requête SQL et le code pandas.

### Problème 1

In [ ]:
sql("""
SELECT age, sex, survived
FROM titanic
WHERE age < 16 AND parch = 0
ORDER BY age
""")

In [ ]:
children = titanic.age < 16
parents = titanic.parch == 0
col_list = ['age', 'sex', 'survived']

titanic.loc[children & parents, col_list].sort_values('age')

### Problème 2

In [ ]:
sql("""
SELECT pclass, age, fare
FROM titanic
WHERE sex = 'female' AND pclass != 1
ORDER BY age DESC, fare DESC
LIMIT 20
""")

In [ ]:
females = titanic.sex == 'female'
lower_class = titanic.pclass != 1
col_list = ['pclass', 'age', 'fare']

df_slice = titanic.loc[females & lower_class, col_list]
df_slice.nlargest(20, ['age', 'fare'])

### Problème 3

In [ ]:
sql("""
SELECT *
FROM titanic
WHERE age < 18 AND pclass = 3 AND survived = 1
ORDER BY age, fare DESC
""")

In [ ]:
children = titanic.age < 18
low_class = titanic.pclass == 3
survival = titanic.survived == 1

df_slice = titanic.loc[children & low_class & survival, :]
df_slice.sort_values(['age', 'fare'], ascending=[True, False])

### Problème 4

In [ ]:
sql("""
SELECT embarked, pclass, COUNT(*), AVG(fare)
FROM titanic
GROUP BY embarked, pclass
""")

In [ ]:
titanic.groupby(['embarked', 'pclass'], dropna=False).agg(
    passagers=('fare', 'size'),
    prix_moyen=('fare', 'mean'),
)

### Problème 5

In [ ]:
sql("""
SELECT pclass, survived, MIN(age), MAX(age), AVG(age)
FROM titanic
GROUP BY pclass, survived
""")

In [ ]:
titanic.groupby(['pclass', 'survived']).age.agg(['min', 'max', 'mean'])

### Problème 6

In [ ]:
sql("""
SELECT survived, AVG(fare), MIN(age), MAX(age)
FROM titanic
GROUP BY survived
""")

In [ ]:
titanic.groupby('survived').agg(
    prix_moyen=('fare', 'mean'),
    age_min=('age', 'min'),
    age_max=('age', 'max'),
)

### Contrôle automatique des 6 corrections

In [ ]:
def meme_resultat(requete, resultat):
    a = sql(requete)
    b = resultat.copy()
    b.columns = a.columns
    # à égalité sur la clé de tri, l'ordre des lignes peut différer : on compare les lignes triées
    a = a.sort_values(list(a.columns)).reset_index(drop=True)
    b = b.sort_values(list(b.columns)).reset_index(drop=True)
    pd.testing.assert_frame_equal(a, b, check_dtype=False)
    return True

print(meme_resultat(
    "SELECT age, sex, survived FROM titanic WHERE age < 16 AND parch = 0 ORDER BY age",
    titanic.loc[(titanic.age < 16) & (titanic.parch == 0), ['age', 'sex', 'survived']].sort_values('age')))

print(meme_resultat(
    "SELECT pclass, age, fare FROM titanic WHERE sex = 'female' AND pclass != 1 ORDER BY age DESC, fare DESC LIMIT 20",
    titanic.loc[(titanic.sex == 'female') & (titanic.pclass != 1), ['pclass', 'age', 'fare']].nlargest(20, ['age', 'fare'])))

print(meme_resultat(
    "SELECT survived, pclass, age, fare FROM titanic WHERE age < 18 AND pclass = 3 AND survived = 1 ORDER BY age, fare DESC",
    titanic.loc[(titanic.age < 18) & (titanic.pclass == 3) & (titanic.survived == 1), ['survived', 'pclass', 'age', 'fare']].sort_values(['age', 'fare'], ascending=[True, False])))

print(meme_resultat(
    "SELECT embarked, pclass, COUNT(*), AVG(fare) FROM titanic GROUP BY embarked, pclass",
    titanic.groupby(['embarked', 'pclass'], dropna=False).agg(passagers=('fare', 'size'), prix_moyen=('fare', 'mean')).reset_index()))

print(meme_resultat(
    "SELECT pclass, survived, MIN(age), MAX(age), AVG(age) FROM titanic GROUP BY pclass, survived",
    titanic.groupby(['pclass', 'survived']).age.agg(['min', 'max', 'mean']).reset_index()))

print(meme_resultat(
    "SELECT survived, AVG(fare), MIN(age), MAX(age) FROM titanic GROUP BY survived",
    titanic.groupby('survived').agg(prix_moyen=('fare', 'mean'), age_min=('age', 'min'), age_max=('age', 'max')).reset_index()))